<a href="https://colab.research.google.com/github/RafiaLaeeq127/Flyrank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [5]:
# Load the lane's slice of the starter data
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")  # use the same file you used in notebooks 01/02

print(df.columns.tolist())
print(df.shape)

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
(30000, 44)


In [6]:
# Unit of analysis: one row = one page
# Show a real slice of the dataframe to demonstrate this
page_level_df = df[['content_id', 'avg_position', 'impressions_90d', 'ctr', 'content_type']].copy()
page_level_df.head(10)

,content_id,avg_position,impressions_90d,ctr,content_type
0,content_304f48230142,10.6,3803,0.76,keyword article
1,content_a1fb4e703a9e,20.3,15320,0.05,keyword article
2,content_9aa793d4d895,36.5,12581,0.09,keyword article
3,content_331d6c4de07b,6.2,11751,0.49,keyword article
4,content_d99b7a2d90ca,44.0,19140,0.13,keyword article
5,content_d4084a4bc775,8.5,3970,0.03,keyword article
6,content_9a34b442b552,7.0,20,0.00,keyword article
7,content_a63219c6e95a,21.2,1724,0.06,keyword article
8,content_5e6c160719bc,46.0,32574,0.09,keyword article
9,content_c27558df2b0c,4.9,1240,0.16,keyword article


In [9]:
# Sketch what the target column would look like
# Simple version: expected CTR based on position, using a rough position-bucket average
df['position_bucket'] = pd.cut(df['avg_position'], bins=[0,3,10,20,50,1000])
expected_ctr_by_bucket = df.groupby('position_bucket', observed=True)['ctr'].transform('mean')

df['expected_ctr'] = expected_ctr_by_bucket
df['opportunity_score'] = df['expected_ctr'] - df['ctr']  # positive = underperforming, high opportunity

# Preview target column
df[['content_id', 'avg_position', 'impressions_90d', 'ctr', 'content_type', 'opportunity_score']].sort_values(
    'opportunity_score', ascending=False
).head(10)

,content_id,avg_position,impressions_90d,ctr,content_type,opportunity_score
12560,content_138e5f3d747a,2.9,17,0.0,keyword article,2.714303
704,content_47667a9af1a6,2.3,4,0.0,feedly article,2.714303
4873,content_38a55f070d18,0.1,10,0.0,keyword article,2.714303
25689,content_ec49cfde89da,3.0,6,0.0,feedly article,2.714303
17010,content_1225954ced40,3.0,2,0.0,keyword article,2.714303
4345,content_119438e47da5,1.0,4,0.0,keyword article,2.714303
5608,content_a55d958ec725,2.7,3,0.0,feedly article,2.714303
10809,content_a138ab3bd14d,1.9,200,0.0,keyword article,2.714303
16953,content_201c5c2257b9,1.5,408,0.0,keyword article,2.714303
6780,content_e80e4dd74da2,1.0,1,0.0,keyword article,2.714303


In [10]:
print(f"""
Observation: The top-ranked pages by opportunity_score have high avg_position
but noticeably lower CTR than similarly-positioned pages. This is a directional
sketch of the target column, not the final model — real feature engineering and
proper regression/ranking modeling will refine this in later weeks.
""")


Observation: The top-ranked pages by opportunity_score have high avg_position 
but noticeably lower CTR than similarly-positioned pages. This is a directional 
sketch of the target column, not the final model — real feature engineering and 
proper regression/ranking modeling will refine this in later weeks.

